In [1]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [2]:
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 25
LR = 1e-4

BASE_PATH = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task"
TRAIN_IMG = os.path.join(BASE_PATH, "train", "images")
TRAIN_MASK = os.path.join(BASE_PATH, "train", "masks")
TEST_IMG = os.path.join(BASE_PATH, "test", "images")
TEST_MASK = os.path.join(BASE_PATH, "test", "masks")

PRED_BASE = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/DeepLabV3MobileNet"
PRED_MASK_DIR = os.path.join(PRED_BASE, "masks")
OVERLAY_DIR = os.path.join(PRED_BASE, "overlays")

MODEL_SAVE_DIR = os.path.join(os.path.dirname(PRED_BASE), "models")
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, "deeplabv3plus_mobilenet.pth")

os.makedirs(PRED_MASK_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

In [3]:
class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.masks = sorted(os.listdir(mask_dir))
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        mask = Image.open(mask_path).convert("L")
        mask = mask.resize((IMG_SIZE, IMG_SIZE))
        mask = np.array(mask).astype("float32") / 255.0
        mask = (mask > 0.5).astype("float32")
        mask = torch.tensor(mask)

        out_name = self.masks[idx]
        return image, mask, out_name


def collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    masks = torch.stack([b[1] for b in batch])
    names = [b[2] for b in batch]
    return images, masks, names

In [4]:
class ASPP(nn.Module):
    def __init__(self, in_channels, out_channels=256):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.conv6 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.conv12 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.conv18 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_channels * 4, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv6(x)
        x3 = self.conv12(x)
        x4 = self.conv18(x)
        x = torch.cat([x1, x2, x3, x4], dim=1)
        return self.project(x)

In [5]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        self.features = mobilenet.features

    def forward(self, x):
        low_level = None
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i == 2:
                low_level = x
        return x, low_level

In [6]:
class Decoder(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.low_conv = nn.Sequential(
            nn.Conv2d(24, 48, kernel_size=1, bias=False),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True)
        )
        self.conv1 = nn.Sequential(
            nn.Conv2d(304, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.classifier = nn.Conv2d(256, num_classes, 1)

    def forward(self, x, low_level):
        low_level = self.low_conv(low_level)
        x = F.interpolate(x, size=low_level.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, low_level], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return self.classifier(x)

In [7]:
class DeepLabV3Plus(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.encoder = Encoder()
        self.aspp = ASPP(1280)
        self.decoder = Decoder(num_classes)

    def forward(self, x):
        x, low_level = self.encoder(x)
        x = self.aspp(x)
        x = self.decoder(x, low_level)
        x = F.interpolate(x, scale_factor=4, mode="bilinear", align_corners=False)
        return x

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepLabV3Plus(num_classes=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print("Device:", device)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /home/shaunakmishra25/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|█████████████████████████████████████████████████████████████████████████████████████| 13.6M/13.6M [00:02<00:00, 5.34MB/s]


Device: cuda


In [9]:
train_dataset = SegDataset(TRAIN_IMG, TRAIN_MASK)
test_dataset = SegDataset(TEST_IMG, TEST_MASK)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

In [10]:
best_val_loss = float('inf')
best_model_path = os.path.join(MODEL_SAVE_DIR, "deeplabv3plus_mobilenet_best.pth")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0

    for images, masks, _ in train_loader:
        images = images.to(device)
        masks = masks.to(device)
        outputs = model(images).squeeze(1)
        loss = criterion(outputs, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, masks, _ in test_loader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images).squeeze(1)
            val_loss += criterion(outputs, masks).item()
    val_loss /= len(test_loader)
    model.train()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)

    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {epoch_loss:.4f} Val Loss: {val_loss:.4f}")

Epoch [1/25] Loss: 0.1040 Val Loss: 0.0411
Epoch [2/25] Loss: 0.0297 Val Loss: 0.0235
Epoch [3/25] Loss: 0.0188 Val Loss: 0.0176
Epoch [4/25] Loss: 0.0143 Val Loss: 0.0166
Epoch [5/25] Loss: 0.0121 Val Loss: 0.0152
Epoch [6/25] Loss: 0.0104 Val Loss: 0.0150
Epoch [7/25] Loss: 0.0093 Val Loss: 0.0133
Epoch [8/25] Loss: 0.0086 Val Loss: 0.0143
Epoch [9/25] Loss: 0.0081 Val Loss: 0.0131
Epoch [10/25] Loss: 0.0075 Val Loss: 0.0133
Epoch [11/25] Loss: 0.0069 Val Loss: 0.0128
Epoch [12/25] Loss: 0.0065 Val Loss: 0.0130
Epoch [13/25] Loss: 0.0063 Val Loss: 0.0137
Epoch [14/25] Loss: 0.0065 Val Loss: 0.0135
Epoch [15/25] Loss: 0.0057 Val Loss: 0.0125
Epoch [16/25] Loss: 0.0055 Val Loss: 0.0136
Epoch [17/25] Loss: 0.0053 Val Loss: 0.0147
Epoch [18/25] Loss: 0.0057 Val Loss: 0.0136
Epoch [19/25] Loss: 0.0052 Val Loss: 0.0154
Epoch [20/25] Loss: 0.0049 Val Loss: 0.0142
Epoch [21/25] Loss: 0.0048 Val Loss: 0.0147
Epoch [22/25] Loss: 0.0047 Val Loss: 0.0144
Epoch [23/25] Loss: 0.0046 Val Loss: 0.01

In [11]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

Model saved to /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/models/deeplabv3plus_mobilenet.pth


In [12]:
model.eval()
torch.set_grad_enabled(False)

with torch.no_grad():
    for images, masks, names in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.sigmoid(outputs)
        preds = (preds > 0.5).float()
        preds = preds.cpu().numpy()
        images_np = images.cpu().numpy()

        for j in range(preds.shape[0]):
            pred_mask = preds[j, 0]
            pred_mask_uint8 = (pred_mask * 255).astype("uint8")

            out_name = names[j]
            mask_path = os.path.join(PRED_MASK_DIR, out_name)
            Image.fromarray(pred_mask_uint8).save(mask_path)

            image_vis = images_np[j].transpose(1, 2, 0)
            overlay = image_vis.copy()
            overlay[pred_mask > 0] = [1, 0, 0]
            combined = 0.7 * image_vis + 0.3 * overlay
            combined = (np.clip(combined, 0, 1) * 255).astype("uint8")
            overlay_path = os.path.join(OVERLAY_DIR, out_name)
            Image.fromarray(combined).save(overlay_path)

print(f"Saved {len(test_dataset)} masks and overlays (filenames match ground truth)")

Saved 860 masks and overlays (filenames match ground truth)


In [13]:
# Load saved model for inference without retraining
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()

DeepLabV3Plus(
  (encoder): Encoder(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU6(inplace=True)
          )
          (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
      (2): InvertedResidual(
        (conv): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 96, kernel_size=(1, 

In [14]:
# Test set metrics (run after inference cell; uses saved predictions from PRED_MASK_DIR)
def dice_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)

def iou_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

def precision_recall_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    return precision, recall

def accuracy_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    return np.sum(y_true == y_pred) / len(y_true)

dice_scores, iou_scores, precision_scores, recall_scores, accuracy_scores = [], [], [], [], []
for i in range(len(test_dataset)):
    _, mask, name = test_dataset[i]
    gt = (mask.numpy() > 0.5).astype(np.uint8)
    pred_path = os.path.join(PRED_MASK_DIR, name)
    pred_img = np.array(Image.open(pred_path))
    pred = (pred_img > 127).astype(np.uint8)
    dice_scores.append(dice_np(gt, pred))
    iou_scores.append(iou_np(gt, pred))
    p, r = precision_recall_np(gt, pred)
    precision_scores.append(p)
    recall_scores.append(r)
    accuracy_scores.append(accuracy_np(gt, pred))

print("===== TEST SET RESULTS =====")
print(f"Mean Dice     : {np.mean(dice_scores):.4f}")
print(f"Mean IoU      : {np.mean(iou_scores):.4f}")
print(f"Mean Precision: {np.mean(precision_scores):.4f}")
print(f"Mean Recall   : {np.mean(recall_scores):.4f}")
print(f"Mean Accuracy : {np.mean(accuracy_scores):.4f}")

===== TEST SET RESULTS =====
Mean Dice     : 0.8432
Mean IoU      : 0.7659
Mean Precision: 0.8486
Mean Recall   : 0.8721
Mean Accuracy : 0.9959


In [15]:
# Ground Truth vs Predicted Comparison and PDF Export
from matplotlib.backends.backend_pdf import PdfPages

SAMPLES_PER_PAGE = 4
COMPARISON_PDF_PATH = os.path.join(PRED_BASE, "comparison_report.pdf")


def make_comparison_overlay(gt_mask, pred_mask):
    """Build color-coded comparison: TP=green, FP=red, FN=blue, TN=black."""
    gt = (gt_mask > 0.5).astype(np.uint8)
    pred = (pred_mask > 0.5).astype(np.uint8)
    h, w = gt.shape
    overlay = np.zeros((h, w, 3), dtype=np.uint8)
    overlay[gt == 1] = [0, 0, 128]      # FN = blue (dark)
    overlay[pred == 1] = [128, 0, 0]   # FP = red (dark)
    overlay[(gt == 1) & (pred == 1)] = [0, 128, 0]  # TP = green
    return overlay


# Include all test images
indices = np.arange(len(test_dataset))

with PdfPages(COMPARISON_PDF_PATH) as pdf:
    for page_start in range(0, len(indices), SAMPLES_PER_PAGE):
        page_indices = indices[page_start : page_start + SAMPLES_PER_PAGE]
        n_rows = len(page_indices)
        fig, axes = plt.subplots(n_rows, 5, figsize=(15, 3 * n_rows))
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        for row, idx in enumerate(page_indices):
            img_name = test_dataset.images[idx]
            mask_name = test_dataset.masks[idx]
            img_path = os.path.join(TEST_IMG, img_name)
            gt_path = os.path.join(TEST_MASK, mask_name)
            pred_path = os.path.join(PRED_MASK_DIR, mask_name)
            overlay_path = os.path.join(OVERLAY_DIR, mask_name)

            image = np.array(Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            gt_mask = np.array(Image.open(gt_path).convert("L").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            pred_mask = np.array(Image.open(pred_path).convert("L")) / 255.0
            if pred_mask.shape != (IMG_SIZE, IMG_SIZE):
                pred_mask = np.array(Image.fromarray((pred_mask * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0

            overlay_img = np.array(Image.open(overlay_path).convert("RGB")) / 255.0
            if overlay_img.shape[:2] != (IMG_SIZE, IMG_SIZE):
                overlay_img = np.array(Image.fromarray((overlay_img * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0

            comparison = make_comparison_overlay(gt_mask, pred_mask)

            axes[row, 0].imshow(image)
            axes[row, 0].set_title("Image")
            axes[row, 0].axis("off")

            axes[row, 1].imshow(gt_mask, cmap="gray")
            axes[row, 1].set_title("Ground Truth")
            axes[row, 1].axis("off")

            axes[row, 2].imshow(pred_mask, cmap="gray")
            axes[row, 2].set_title("Predicted")
            axes[row, 2].axis("off")

            axes[row, 3].imshow(overlay_img)
            axes[row, 3].set_title("Overlay")
            axes[row, 3].axis("off")

            axes[row, 4].imshow(comparison) 
            axes[row, 4].set_title("Comparison (TP=green, FP=red, FN=blue)")
            axes[row, 4].axis("off")

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Comparison PDF saved to {COMPARISON_PDF_PATH}")

Comparison PDF saved to /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/DeepLabV3EfficientNetB0/comparison_report.pdf
